# Bayesmark benchmark score barplot

This notebook reproduces the Bayesmark barplot used in the original analysis code.
It expects the benchmark outputs to be available in `results/part1`, ..., `results/part20`
and to have been processed with `python run_benchmark.py aggregate`.

The plotted score is the same as in the historical notebook: `100 * (1 - PERF_MEAN)`
at optimization iteration 99, averaged by `seaborn.barplot` across the 20 benchmark repeats.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import bayesmark.constants as cc
from bayesmark.constants import ITER, OBJECTIVE, VISIBLE_TO_OPT
from bayesmark.serialize import XRSerializer


## Settings

Change `RESULTS_DIR` or `N_REPEATS` only if the downloaded/generated result directory
uses a different location or number of benchmark repeats.


In [ ]:
RESULTS_DIR = Path("results").resolve()
N_REPEATS = 20
# For the smoke test, use Path("smoke_results").resolve() and N_REPEATS = 2.
FINAL_ITERATION = 99
metric_for_scoring = VISIBLE_TO_OPT

PAPER_PLOT_ORDER = ["HEBO", "tidyHEBO", "LogEI", "random"]
PALETTE = {
    "HEBO": "tab:purple",
    "LogEI": "tab:orange",
    "tidyHEBO": "tab:red",
    "random": "tab:blue",
    "optuna": "tab:green",
}

if not RESULTS_DIR.exists():
    raise FileNotFoundError(
        f"{RESULTS_DIR} does not exist. Generate the benchmark results or place "
        "the Zenodo result archive in ./results first."
    )


In [ ]:
def get_df(repeat: int) -> pd.DataFrame:
    """Load the final Bayesmark mean score for one benchmark repeat."""
    summary_ds, _ = XRSerializer.load_derived(
        str(RESULTS_DIR), db=f"part{repeat}", key=cc.MEAN_SCORE
    )

    scores = 100 * (
        1
        - summary_ds[cc.PERF_MEAN].sel(
            {ITER: FINAL_ITERATION, OBJECTIVE: metric_for_scoring}
        )
    )

    df = pd.DataFrame(
        np.vstack([scores.optimizer.values, scores.values]).T,
        columns=["optimizer", "score"],
    )
    df["score"] = pd.to_numeric(df["score"], errors="coerce")
    df["optimizer"] = df["optimizer"].str.split("_").str.get(0)
    df.loc[df["optimizer"] == "RandomSearch", "optimizer"] = "random"
    df["repeat"] = repeat
    return df


In [ ]:
df = pd.concat(
    [get_df(repeat) for repeat in range(1, N_REPEATS + 1)],
    ignore_index=True,
)

available = set(df["optimizer"].dropna().unique())
PLOT_ORDER = [name for name in PAPER_PLOT_ORDER if name in available]
if not PLOT_ORDER and "optuna" in available:
    PLOT_ORDER = ["optuna"]

df[df["optimizer"].isin(PLOT_ORDER)].head(10)


## Barplot

For the full benchmark, the plot follows the historical optimizer selection. If the notebook is pointed at `smoke_results/`, it shows Optuna as a smoke-test check.


In [ ]:
plt.rcdefaults()

ax = sns.barplot(
    data=df,
    x="optimizer",
    y="score",
    order=PLOT_ORDER,
    hue="optimizer",
    palette=PALETTE,
    saturation=1,
)

ax.set_ylim(bottom=75, top=100)

FNT_SIZE = 13
ax.set_xlabel("optimizer", fontsize=FNT_SIZE)
ax.set_ylabel("bayesmark averaged score", fontsize=FNT_SIZE)
plt.xticks(fontsize=FNT_SIZE - 1)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=15, fontsize=FNT_SIZE)

ax.figure.tight_layout()
plt.show()
